# Evaluate Cp Relative L2 Error

This notebook loads a trained checkpoint (e.g. `checkpoints/best.pt`) and computes **Cp relative L2 error** on the selected split.

- Uses the same Cp definition as `scripts/train.py` (Cp = (p/ρ) / q, q = 0.5 * U∞²).
- Computes metrics on **surface nodes** by default.


In [1]:
import os
import sys
import math
from typing import Iterable

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Ensure repo root is on PYTHONPATH
repo_root = os.path.abspath(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.training_common import SmokeCfg, load_and_prepare_data, collate_pyg, NormalizedDataset
from src.global_context_processor import EnhancedCFDModelWithGlobalContext
from src.utils import _prep_graph_for_norm

torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- User settings ----
CKPT_PATH = "checkpoints/best.pt"   # path to your trained checkpoint
SPLIT = "val"                      # "val" (training val) or "test" (prebuilt_edges_v2/<task>/test)
SURFACE_ONLY = True                # evaluate Cp error on surface nodes only
EPS = 1e-12

# Must match your training run
scfg = SmokeCfg()
# scfg.task = "scarce"             # optionally override
# scfg.hidden = 128                 # must match checkpoint
# scfg.layers = 14                  # must match checkpoint

print("task:", scfg.task)


/home/elicer/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda
task: scarce


In [2]:
def _surface_mask_from_x_phys(x_phys: torch.Tensor) -> torch.Tensor:
    # Mirrors src/utils.py:get_surface_mask(d) but operates on *physical* x
    if x_phys.dim() != 2:
        raise ValueError(f"x_phys must be 2D [N,F], got {tuple(x_phys.shape)}")
    if x_phys.size(1) >= 5:
        wall = x_phys[:, 2]
        nxy = x_phys[:, 3:5]
        return (wall < 1e-6) | (nxy.abs().sum(dim=1) > 0)
    if x_phys.size(1) >= 3:
        wall = x_phys[:, 2]
        return (wall < 1e-6)
    return torch.zeros(x_phys.size(0), dtype=torch.bool)


def _cp_from_p_over_rho(y_phys: torch.Tensor, x_phys: torch.Tensor, *, p_channel: int = 2, eps: float = 1e-12) -> torch.Tensor:
    # Matches scripts/train.py: q = 0.5 * ||(u_inf, v_inf)||^2, Cp = (p/ρ)/q
    if y_phys.dim() != 2 or x_phys.dim() != 2:
        raise ValueError("y_phys and x_phys must be [N, C] and [N, F]")
    if x_phys.size(1) < 2:
        raise ValueError("x_phys must have at least 2 columns for (u_inf, v_inf)")

    u_inf = x_phys[0, 0]
    v_inf = x_phys[0, 1]
    q = 0.5 * (u_inf * u_inf + v_inf * v_inf)
    q = q.clamp_min(eps)
    return y_phys[:, p_channel] / q


def _relative_l2(pred: torch.Tensor, targ: torch.Tensor, *, eps: float = 1e-12) -> float:
    pred = pred.reshape(-1)
    targ = targ.reshape(-1)
    if pred.numel() == 0:
        return float("nan")
    num = torch.linalg.norm(pred - targ)
    den = torch.linalg.norm(targ).clamp_min(eps)
    return float((num / den).item())


In [3]:
# ---- Load data bundle (scalers + graphs) ----
if not os.path.isdir(os.path.join("prebuilt_edges_v2", scfg.task)):
    raise FileNotFoundError(
        f"Missing prebuilt edges at prebuilt_edges_v2/{scfg.task}. "
        "Run preprocessing first (see CLAUDE.md commands)."
    )

data_bundle = load_and_prepare_data(scfg)
x_scaler = data_bundle.x_scaler
y_scaler = data_bundle.y_scaler

if SPLIT == "val":
    if not isinstance(data_bundle.val_norm, NormalizedDataset):
        raise ValueError("No val dataset available (data_bundle.val_norm is empty)")
    eval_ds = data_bundle.val_norm
elif SPLIT == "test":
    test_graphs = data_bundle.val_graphs
    if not isinstance(test_graphs, list) or len(test_graphs) == 0:
        raise ValueError("No test graphs available (data_bundle.val_graphs is empty)")
    test_prepped = [_prep_graph_for_norm(g) for g in test_graphs]
    eval_ds = NormalizedDataset(test_prepped, x_scaler, y_scaler)
else:
    raise ValueError("SPLIT must be 'val' or 'test'")

eval_loader = DataLoader(eval_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_pyg)
print(f"eval split={SPLIT} | graphs={len(eval_ds)}")


[prebuilt] found: 200 train and 200 val graphs under prebuilt_edges_v2/scarce
Graphs prepared. Example dims -> x: torch.Size([17987, 5])  edge_attr: torch.Size([68640, 5])
[validate] train_edges: total=200 bad=0
Prepared normalized datasets: 180 train | 20 val
eval split=val | graphs=20


In [4]:
# ---- Load model + checkpoint ----
node_dim = 7
edge_dim = 5

model = EnhancedCFDModelWithGlobalContext(
    node_feat_dim=node_dim,
    edge_feat_dim=edge_dim,
    hidden_dim=scfg.hidden,
    output_dim=4,
    num_mp_layers=scfg.layers,
    dropout_p=0.1,
    config=scfg,
).to(device)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
model.load_state_dict(state)
model.eval()

print("Loaded:", CKPT_PATH)


Loaded: checkpoints/best.pt


In [5]:
# ---- Compute Cp relative L2 error ----
per_graph = []
sum_err2 = 0.0
sum_gt2 = 0.0

for batch in tqdm(eval_loader, desc="eval"):
    if batch is None:
        continue
    b = batch.to(device)

    pred_norm = model(b).detach().cpu()
    targ_norm = b.y.detach().cpu()

    pred_phys = y_scaler.inverse(pred_norm)
    targ_phys = y_scaler.inverse(targ_norm)
    x_phys = x_scaler.inverse(b.x.detach().cpu())

    cp_pred = _cp_from_p_over_rho(pred_phys, x_phys, p_channel=2, eps=EPS)
    cp_true = _cp_from_p_over_rho(targ_phys, x_phys, p_channel=2, eps=EPS)

    if SURFACE_ONLY:
        m = _surface_mask_from_x_phys(x_phys)
        cp_pred = cp_pred[m]
        cp_true = cp_true[m]

    rel = _relative_l2(cp_pred, cp_true, eps=EPS)
    per_graph.append(rel)

    err = (cp_pred - cp_true)
    sum_err2 += float((err * err).sum().item())
    sum_gt2 += float((cp_true * cp_true).sum().item())

per_graph = np.asarray(per_graph, dtype=np.float64)
global_rel = math.sqrt(sum_err2 / (sum_gt2 + EPS))

print("\nCp relative L2 error")
print("- split:", SPLIT)
print("- surface_only:", SURFACE_ONLY)
print("- graphs:", int(np.isfinite(per_graph).sum()), "/", len(per_graph))
print(f"- mean   : {np.nanmean(per_graph):.6g}")
print(f"- median : {np.nanmedian(per_graph):.6g}")
print(f"- std    : {np.nanstd(per_graph):.6g}")
print(f"- global : {global_rel:.6g}   (aggregated over all evaluated nodes)")


eval: 100%|██████████| 20/20 [00:01<00:00, 14.80it/s]


Cp relative L2 error
- split: val
- surface_only: True
- graphs: 20 / 20
- mean   : 0.792348
- median : 0.826013
- std    : 0.166997
- global : 0.597794   (aggregated over all evaluated nodes)
